In [1]:
import os
import copy
import importlib
import numpy as np

import gwfast_mod.gwfastGlobals as glob
import gwfast_mod.waveforms as waveforms
import gwfast_mod.signal as signal
import gwfast_mod.network as network
import gwfast_mod.fisherTools as fTools

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/gwfast_mod/waveforms.py:33: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [2]:
# importlib.reload(signal)

In [3]:
events = {
    'Mc':np.array([50]), 'eta':np.array([0.24]), 'dL':np.array([0.8]), 'theta':np.array([2.34]), 'phi':np.array([5.43]), 
    'iota':np.array([0.9 * np.pi/2]), 'psi':np.array([1]), 'tGPS':np.array([0]), 'Phicoal':np.array([2.8]), 
    'chi1z':np.array([1e-3]), 'chi2z':np.array([1e-3]), 'chi1x':np.array([0]), 'chi2x':np.array([0]), 'chi1y':np.array([0]), 'chi2y':np.array([0]), 
    'Lambda1':np.array([0]), 'Lambda2':np.array([0]), 'ecc':np.array([0]), 
    'phi_L':np.array([0.9*np.pi/2]), 'R_orbit':np.array([20])
}

mywf = waveforms.TaylorF2_RestrictedPN()

In [4]:
L1_conf = copy.deepcopy(glob.detectors).pop('L1')
L1_conf['psd_path'] = os.path.join(glob.detPath, 'observing_scenarios_paper', 'AplusDesign.txt')

H1_conf = copy.deepcopy(glob.detectors).pop('H1')
H1_conf['psd_path'] = os.path.join(glob.detPath, 'observing_scenarios_paper', 'AplusDesign.txt')

Virgo_conf = copy.deepcopy(glob.detectors).pop('Virgo')
Virgo_conf['psd_path'] =  os.path.join(glob.detPath, 'observing_scenarios_paper', 'avirgo_O5low_NEW.txt')

# Initialise the GWSignal objects
L1 = signal.GWSignal(mywf, psd_path=L1_conf['psd_path'], 
                        detector_shape=L1_conf['shape'], det_lat=L1_conf['lat'], 
                        det_long=L1_conf['long'], det_xax=L1_conf['xax'],
                        fmin=10
                        )
H1 = signal.GWSignal(mywf, psd_path=H1_conf['psd_path'], 
                        detector_shape=H1_conf['shape'], det_lat=H1_conf['lat'], 
                        det_long=H1_conf['long'], det_xax=H1_conf['xax'],
                        fmin=10
                        )
Virgo = signal.GWSignal(mywf, psd_path=Virgo_conf['psd_path'], 
                        detector_shape=Virgo_conf['shape'], det_lat=Virgo_conf['lat'], 
                        det_long=Virgo_conf['long'], det_xax=Virgo_conf['xax'],
                        fmin=10
                        )

mySignals = {'L1':L1, 'H1':H1, 'Virgo':Virgo}
myNet = network.DetNet(mySignals)

Using ASD from file /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/psds/observing_scenarios_paper/AplusDesign.txt 
Initializing jax...
Jax local device count: 8
Jax  device count: 8


Using ASD from file /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/psds/observing_scenarios_paper/AplusDesign.txt 
Initializing jax...
Jax local device count: 8
Jax  device count: 8
Using ASD from file /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/psds/observing_scenarios_paper/avirgo_O5low_NEW.txt 
Initializing jax...
Jax local device count: 8
Jax  device count: 8


In [5]:
mySNR = myNet.SNR(events, use_lensing=True, return_all=True)
mySNR_nolens = myNet.SNR(events, return_all=True)
print("SNR with lensing:", mySNR)
print("SNR without lensing:", mySNR_nolens)

Adding tcoal from tGPS
{'Mc': Array([43.88183383], dtype=float64), 'eta': array([0.24]), 'dL': array([0.8]), 'theta': array([2.34]), 'phi': array([5.43]), 'iota': Array([1.42154463], dtype=float64), 'psi': Array([1.04930972], dtype=float64), 'tGPS': array([0]), 'Phicoal': Array([2.55893316], dtype=float64), 'chi1z': array([0.001]), 'chi2z': array([0.001]), 'chi1x': array([0]), 'chi2x': array([0]), 'chi1y': array([0]), 'chi2y': array([0]), 'Lambda1': array([0]), 'Lambda2': array([0]), 'ecc': array([0]), 'phi_L': array([1.41371669]), 'R_orbit': array([20]), 'tcoal': Array([0.29095751], dtype=float64), 'LambdaTilde': Array([0.], dtype=float64), 'deltaLambda': Array([-0.], dtype=float64)}
{'Mc': Array([43.88183383], dtype=float64), 'eta': array([0.24]), 'dL': array([0.8]), 'theta': array([2.34]), 'phi': array([5.43]), 'iota': Array([1.42154463], dtype=float64), 'psi': Array([1.04930972], dtype=float64), 'tGPS': array([0]), 'Phicoal': Array([2.55893316], dtype=float64), 'chi1z': array([0.00

In [6]:
FisherMatrsNet = myNet.FisherMatr(events, use_lensing=True)
FisherMatrsNet

Computing Fisher for L1...
Computing derivatives...
Computing Fisher for H1...
Computing derivatives...
Computing Fisher for Virgo...
Computing derivatives...
Done.


array([[[ 1.75210624e+01],
        [-3.65464514e+03],
        [ 5.77936424e+00],
        [ 1.60772690e+01],
        [-2.45867993e+01],
        [-1.06176928e+01],
        [-6.32510789e+00],
        [ 1.92895358e+03],
        [-2.34429854e+01],
        [-1.36584256e+03],
        [-8.48092270e+02],
        [-8.11929363e-01],
        [ 3.41937896e+00]],

       [[-3.65464514e+03],
        [ 2.56510143e+06],
        [-3.47880165e+03],
        [-2.38050751e+04],
        [ 3.44188306e+04],
        [ 5.53259040e+03],
        [ 1.07805751e+04],
        [-4.60294436e+06],
        [ 1.68520334e+04],
        [ 1.18667658e+06],
        [ 7.76311578e+05],
        [ 8.41174436e+01],
        [-9.61115561e+02]],

       [[ 5.77936424e+00],
        [-3.47880165e+03],
        [ 4.44920151e+02],
        [ 5.12736896e+02],
        [ 3.18955722e+02],
        [ 2.39869679e+02],
        [ 8.50696325e+01],
        [ 2.49548163e+03],
        [-5.90347506e+01],
        [-1.27481070e+03],
        [-7.67505918e+02

In [7]:
FisherMatrsNet_nolens = myNet.FisherMatr(events)
FisherMatrsNet_nolens

Computing Fisher for L1...
Computing derivatives...
Computing Fisher for H1...
Computing derivatives...
Computing Fisher for Virgo...
Computing derivatives...
Done.


array([[[ 9.51731309e+00],
        [-2.08691201e+03],
        [-3.22935664e+00],
        [-1.65700428e+01],
        [-2.03170663e+01],
        [ 9.10541945e+00],
        [-1.83892030e+01],
        [ 8.87491164e+02],
        [-1.10991445e+01],
        [-7.83160887e+02],
        [-4.86459403e+02]],

       [[-2.08691201e+03],
        [ 1.43359364e+06],
        [ 3.06017418e-14],
        [ 3.04023403e+03],
        [ 2.26987972e+04],
        [-5.78157801e+03],
        [ 1.53470508e+04],
        [-2.50001245e+06],
        [ 1.41376225e+04],
        [ 6.60846431e+05],
        [ 4.31839977e+05]],

       [[-3.22935664e+00],
        [ 3.06017418e-14],
        [ 2.42201748e+02],
        [ 2.75925270e+02],
        [ 1.98913471e+02],
        [ 1.39667599e+02],
        [ 6.46261672e+01],
        [-1.44653958e-02],
        [ 4.78301286e-16],
        [ 2.41636640e-14],
        [ 8.09739154e-15]],

       [[-1.65700428e+01],
        [ 3.04023403e+03],
        [ 2.75925270e+02],
        [ 1.37244379e+

In [8]:
Cov, ie = fTools.CovMatr(FisherMatrsNet)
Cov_nolens, ie_nolens = fTools.CovMatr(FisherMatrsNet_nolens)
ie, ie_nolens

(array([1.02600684e-09], dtype=float128), array([0.00487518], dtype=float128))

In [9]:
print((np.diag(Cov[:,:,0])))
print(np.sqrt(np.diag(Cov_nolens[:,:,0])))

[8.57670619e+00 1.54189608e-02 1.13303618e-02 1.84751457e-03
 3.72523440e-03 2.07989857e-03 1.19829315e-02 8.91183591e-04
 1.92243135e+00 3.21918135e+00 7.31272913e+00 5.16458962e-01
 4.30025246e+00]
[3.72989573e+02 1.47379289e+02 4.97402807e+00 6.68042222e-02
 9.96231419e-02 5.65416934e-02 1.30076439e-01 2.41854095e+01
 3.48811452e+04 1.28534475e+03 3.45698361e+03]


In [10]:
Cov

array([[[ 8.57670619e+00],
        [ 3.19921733e-01],
        [ 1.38297601e-01],
        [ 6.53539162e-03],
        [ 1.13587888e-02],
        [ 8.23891795e-03],
        [-2.34546280e-02],
        [-4.17294658e-02],
        [-1.95319285e+00],
        [-3.74253813e-01],
        [-6.52969840e-01],
        [ 4.41213934e-01],
        [-3.11320613e+00]],

       [[ 3.19921733e-01],
        [ 1.54189608e-02],
        [ 2.25526585e-03],
        [ 3.11264094e-04],
        [ 5.05702410e-04],
        [ 2.37416558e-04],
        [-1.21553939e-03],
        [-4.61000458e-04],
        [-1.86017384e-02],
        [-1.05990688e-01],
        [ 1.10227691e-01],
        [ 7.24627685e-03],
        [-2.62032116e-02]],

       [[ 1.38297601e-01],
        [ 2.25526585e-03],
        [ 1.13303618e-02],
        [-1.58894849e-03],
        [-1.41156787e-03],
        [-5.16087844e-04],
        [ 3.71153404e-03],
        [-2.23495080e-03],
        [-1.00647118e-01],
        [ 1.01997217e-01],
        [-1.73937890e-01

In [11]:
!which python3

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


/Library/Frameworks/Python.framework/Versions/3.10/bin/python3
